# IMPORT LIBRARY

In [7]:
!pip install gensim
import sympy
import sympy.printing
import pandas as pd
import re
from transformers import pipeline
import nltk
from nltk.corpus import stopwords
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from gensim.models import Word2Vec
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import joblib
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# IMPORT DATA

In [8]:
df = pd.read_csv("/content/dataset (1).csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14999 entries, 0 to 14998
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              14999 non-null  object
 1   userName              14999 non-null  object
 2   userImage             14999 non-null  object
 3   content               14999 non-null  object
 4   score                 14999 non-null  int64 
 5   thumbsUpCount         14999 non-null  int64 
 6   reviewCreatedVersion  14995 non-null  object
 7   at                    14999 non-null  object
 8   replyContent          14002 non-null  object
 9   repliedAt             14002 non-null  object
 10  appVersion            14995 non-null  object
dtypes: int64(2), object(9)
memory usage: 1.3+ MB


In [9]:
df = df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 13998 entries, 0 to 14998
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype 
---  ------                --------------  ----- 
 0   reviewId              13998 non-null  object
 1   userName              13998 non-null  object
 2   userImage             13998 non-null  object
 3   content               13998 non-null  object
 4   score                 13998 non-null  int64 
 5   thumbsUpCount         13998 non-null  int64 
 6   reviewCreatedVersion  13998 non-null  object
 7   at                    13998 non-null  object
 8   replyContent          13998 non-null  object
 9   repliedAt             13998 non-null  object
 10  appVersion            13998 non-null  object
dtypes: int64(2), object(9)
memory usage: 1.3+ MB


# PELABELAN

In [10]:

df['label_score'] = df['score'].apply(
    lambda x: 'negative' if x <= 2 else ('neutral' if x == 3 else 'positive')
)

def preprocess(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['content_clean'] = df['content'].apply(preprocess)

classifier = pipeline(
    "text-classification",
    model="mdhugol/indonesia-bert-sentiment-classification",
    truncation=True,
    max_length=512,
    device=0
)

label_map = {'LABEL_0': 'positive', 'LABEL_1': 'neutral', 'LABEL_2': 'negative'}

results = classifier(df['content_clean'].tolist(), batch_size=64)
df['label_model'] = [label_map[r['label']] for r in results]
df['model_confidence'] = [r['score'] for r in results]

def hybrid_label(row):
    if row['label_score'] == row['label_model']:
        return row['label_score']
    elif row['model_confidence'] >= 0.85:
        return row['label_model']
    else:
        return row['label_score']

df['sentiment'] = df.apply(hybrid_label, axis=1)
df['is_conflict'] = df['label_score'] != df['label_model']

print(df['sentiment'].value_counts())

print(f"\n=== Konflik score vs model ===")
print(f"Total konflik : {df['is_conflict'].sum()}")
print(f"Persentase    : {df['is_conflict'].mean()*100:.1f}%")

print(df[df['is_conflict']][['content', 'score', 'label_score', 'label_model', 'model_confidence', 'sentiment']].head(10))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: mdhugol/indonesia-bert-sentiment-classification
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


sentiment
negative    7895
positive    5545
neutral      558
Name: count, dtype: int64

=== Konflik score vs model ===
Total konflik : 3683
Persentase    : 26.3%
                                              content  score label_score  \
0   Asli keamanannya bagus, aman juga lancar semua...      5    positive   
1   Top,Saya sangat suka belanja disini selain ban...      5    positive   
4   koneksinya padahal bagus tapi susah banget bua...      5    positive   
7   sama seperti user lain,aplikasi lemot sekali,p...      5    positive   
11  apk nya udh bagus, tpi cuma mau ksh saran, fit...      4    positive   
13  aplikasi nya bagus, tapi hilangin notifikasi t...      3     neutral   
21  Sebenarnya ini aplikasi sangat bagus, harga be...      4    positive   
22  toko² di shoppe bisa dipertanggungjawabkan kar...      5    positive   
28  perusahaan sebesar shope tapi apk nya berat ba...      5    positive   
34  Aplikasi Shopeefood nya dari dulu gak berubah....      1    negative   

 

In [12]:
df_final = df[~df['is_conflict']].reset_index(drop=True)

print(f"Data sebelum : {len(df)}")
print(f"Data sesudah : {len(df_final)}")
print(f"Dihapus      : {df['is_conflict'].sum()}")
print(f"\nDistribusi sentiment:")
print(df_final['sentiment'].value_counts())
print(f"\nDistribusi score:")
print(df_final['score'].value_counts())

Data sebelum : 13998
Data sesudah : 10315
Dihapus      : 3683

Distribusi sentiment:
sentiment
negative    5547
positive    4673
neutral       95
Name: count, dtype: int64

Distribusi score:
score
1    4567
5    4426
2     980
4     247
3      95
Name: count, dtype: int64


# PRE PROCESSING TEKS

In [13]:
nltk.download('stopwords')
stop_words = set(stopwords.words('indonesian'))
url = "https://raw.githubusercontent.com/nasalsabila/kamus-alay/master/colloquial-indonesian-lexicon.csv"
kamus_df = pd.read_csv(url)
slang_dict = dict(zip(kamus_df['slang'], kamus_df['formal']))
print(f"Total kamus slang: {len(slang_dict)} kata")

def normalize_slang(text):
    return ' '.join([slang_dict.get(w, w) for w in text.split()])
def preprocess_full(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = normalize_slang(text)
    text = ' '.join([w for w in text.split() if w not in stop_words])
    return text
def preprocess_bert(text):
    text = str(text)
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    text = re.sub(r'[^\w\s\.,!?]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df_final['content_clean'] = df_final['content'].apply(preprocess_full)
df_final['content_bert']  = df_final['content'].apply(preprocess_bert)

print("=== Original ===")
print(df_final['content'].iloc[0])
print("\n=== Clean (SVM/RF) ===")
print(df_final['content_clean'].iloc[0])
print("\n=== BERT ===")
print(df_final['content_bert'].iloc[0])

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Total kamus slang: 4331 kata
=== Original ===
Banyak error, ya ampun lemot banget terutama bagian qris shopeepay nya. Masalah jaringan terus, padahal wifi atau paket data kenceng banget. Kadang foto juga ngga update dari galeri, please fix this shopee. Ratingnya banyak banget yang buzzer, padahal aplikasinya masih cukup kurang di beberapa hal.

=== Clean (SVM/RF) ===
error ya ampun lemot banget qris shopeepay nya jaringan wifi paket data kencang banget kadang foto update galeri please fix this shopee ratingnya banget buzzer aplikasinya

=== BERT ===
Banyak error, ya ampun lemot banget terutama bagian qris shopeepay nya. Masalah jaringan terus, padahal wifi atau paket data kenceng banget. Kadang foto juga ngga update dari galeri, please fix this shopee. Ratingnya banyak banget yang buzzer, padahal aplikasinya masih cukup kurang di beberapa hal.


# ENCODE & SPLIT

In [14]:
le = LabelEncoder()
df_final['label'] = le.fit_transform(df_final['sentiment'])

print("Mapping label:")
print(dict(zip(le.classes_, le.transform(le.classes_))))

X_train, X_test, y_train, y_test = train_test_split(
    df_final['content_clean'], df_final['label'],
    test_size=0.2, random_state=42, stratify=df_final['label']
)

X_train_bert, X_test_bert, y_train_bert, y_test_bert = train_test_split(
    df_final['content_bert'], df_final['label'],
    test_size=0.2, random_state=42, stratify=df_final['label']
)

print(f"\nTrain : {len(X_train)}")
print(f"Test  : {len(X_test)}")
print(f"\nDistribusi y_train:")
print(pd.Series(y_train).value_counts())
print(f"\nDistribusi y_test:")
print(pd.Series(y_test).value_counts())

Mapping label:
{'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}

Train : 8252
Test  : 2063

Distribusi y_train:
label
0    4438
2    3738
1      76
Name: count, dtype: int64

Distribusi y_test:
label
0    1109
2     935
1      19
Name: count, dtype: int64


# SVM + TF-IDF

In [15]:
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

svm = LinearSVC(random_state=42, max_iter=1000)
svm.fit(X_train_tfidf, y_train)

y_pred_train = svm.predict(X_train_tfidf)
y_pred_test  = svm.predict(X_test_tfidf)

print("=== Skema 1: SVM + TF-IDF ===")
print(f"Akurasi Train : {accuracy_score(y_train, y_pred_train):.4f}")
print(f"Akurasi Test  : {accuracy_score(y_test, y_pred_test):.4f}")
print("\nClassification Report (Test):")
print(classification_report(y_test, y_pred_test, target_names=le.classes_))

=== Skema 1: SVM + TF-IDF ===
Akurasi Train : 0.9960
Akurasi Test  : 0.9462

Classification Report (Test):
              precision    recall  f1-score   support

    negative       0.94      0.96      0.95      1109
     neutral       0.00      0.00      0.00        19
    positive       0.95      0.95      0.95       935

    accuracy                           0.95      2063
   macro avg       0.63      0.64      0.63      2063
weighted avg       0.94      0.95      0.94      2063



# RF + WORD2VEC

In [16]:
X_train_tokens = [text.split() for text in X_train]
X_test_tokens  = [text.split() for text in X_test]

w2v = Word2Vec(
    sentences=X_train_tokens,
    vector_size=100,
    window=5,
    min_count=2,
    workers=4,
    seed=42
)

def doc_to_vector(tokens, model):
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

X_train_w2v = np.array([doc_to_vector(t, w2v) for t in X_train_tokens])
X_test_w2v  = np.array([doc_to_vector(t, w2v) for t in X_test_tokens])

rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train_w2v, y_train)

y_pred_train_rf = rf.predict(X_train_w2v)
y_pred_test_rf  = rf.predict(X_test_w2v)

print("=== Skema 2: RF + Word2Vec ===")
print(f"Akurasi Train : {accuracy_score(y_train, y_pred_train_rf):.4f}")
print(f"Akurasi Test  : {accuracy_score(y_test, y_pred_test_rf):.4f}")
print("\nClassification Report (Test):")
print(classification_report(y_test, y_pred_test_rf, target_names=le.classes_))

=== Skema 2: RF + Word2Vec ===
Akurasi Train : 1.0000
Akurasi Test  : 0.9157

Classification Report (Test):
              precision    recall  f1-score   support

    negative       0.91      0.94      0.92      1109
     neutral       0.00      0.00      0.00        19
    positive       0.92      0.90      0.91       935

    accuracy                           0.92      2063
   macro avg       0.61      0.62      0.61      2063
weighted avg       0.91      0.92      0.91      2063



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


# INDOBERT

In [17]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

MODEL_NAME = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3)
model.to(device)

class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = texts.reset_index(drop=True)
        self.labels    = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids'      : encoding['input_ids'].squeeze(),
            'attention_mask' : encoding['attention_mask'].squeeze(),
            'label'          : torch.tensor(self.labels[idx], dtype=torch.long)
        }

BATCH_SIZE = 32

train_dataset = SentimentDataset(X_train_bert, y_train_bert, tokenizer)
test_dataset  = SentimentDataset(X_test_bert,  y_test_bert,  tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

EPOCHS    = 3
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=len(train_loader),
    num_training_steps=len(train_loader) * EPOCHS
)

def train_epoch(model, loader, optimizer, scheduler):
    model.train()
    total_loss, preds_all, labels_all = 0, [], []

    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['label'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss    = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds_all  += torch.argmax(outputs.logits, dim=1).cpu().tolist()
        labels_all += labels.cpu().tolist()

    return total_loss / len(loader), accuracy_score(labels_all, preds_all)

def eval_epoch(model, loader):
    model.eval()
    preds_all, labels_all = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels         = batch['label'].to(device)

            outputs    = model(input_ids=input_ids, attention_mask=attention_mask)
            preds_all  += torch.argmax(outputs.logits, dim=1).cpu().tolist()
            labels_all += labels.cpu().tolist()

    return accuracy_score(labels_all, preds_all), preds_all, labels_all

print("=== Skema 3: IndoBERT ===")
for epoch in range(EPOCHS):
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler)
    test_acc, preds, labels = eval_epoch(model, test_loader)

    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Test Acc: {test_acc:.4f}")

print("\nClassification Report (Test):")
print(classification_report(labels, preds, target_names=le.classes_))

Device: cuda


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/498M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/498M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: indobenchmark/indobert-base-p1
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


=== Skema 3: IndoBERT ===
Epoch 1/3 | Loss: 0.2908 | Train Acc: 0.8986 | Test Acc: 0.9636
Epoch 2/3 | Loss: 0.0833 | Train Acc: 0.9753 | Test Acc: 0.9714
Epoch 3/3 | Loss: 0.0252 | Train Acc: 0.9922 | Test Acc: 0.9738

Classification Report (Test):
              precision    recall  f1-score   support

    negative       0.98      0.98      0.98      1109
     neutral       0.38      0.26      0.31        19
    positive       0.98      0.98      0.98       935

    accuracy                           0.97      2063
   macro avg       0.78      0.74      0.76      2063
weighted avg       0.97      0.97      0.97      2063



# INFFERENCE

In [18]:
joblib.dump(svm,   'model_svm.pkl')
joblib.dump(tfidf, 'tfidf_vectorizer.pkl')
joblib.dump(rf,    'model_rf.pkl')
joblib.dump(w2v,   'word2vec.model')
joblib.dump(le,    'label_encoder.pkl')

model.save_pretrained('model_indobert')
tokenizer.save_pretrained('model_indobert')

print("Semua model tersimpan ✅")

def predict_svm(text):
    clean = preprocess_full(text)
    vec   = tfidf.transform([clean])
    pred  = svm.predict(vec)[0]
    return le.inverse_transform([pred])[0]

def predict_rf(text):
    clean  = preprocess_full(text)
    tokens = clean.split()
    vec    = doc_to_vector(tokens, w2v).reshape(1, -1)
    pred   = rf.predict(vec)[0]
    return le.inverse_transform([pred])[0]

def predict_bert(text):
    model.eval()
    clean    = preprocess_bert(text)
    encoding = tokenizer(
        clean,
        max_length=128,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )
    input_ids      = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        pred    = torch.argmax(outputs.logits, dim=1).item()

    return le.inverse_transform([pred])[0]

test_sentences = [
    "aplikasi sangat bagus dan mudah digunakan, pengiriman cepat!",
    "aplikasi sering error dan lambat, sangat mengecewakan",
    "biasa saja, tidak ada yang spesial dari aplikasi ini"
]

print("=" * 55)
print(f"{'Teks':<40} {'SVM':<10} {'RF':<10} {'BERT':<10}")
print("=" * 55)
for text in test_sentences:
    print(f"{text[:38]:<40} {predict_svm(text):<10} {predict_rf(text):<10} {predict_bert(text):<10}")
print("=" * 55)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Semua model tersimpan ✅
Teks                                     SVM        RF         BERT      
aplikasi sangat bagus dan mudah diguna   positive   positive   positive  
aplikasi sering error dan lambat, sang   negative   negative   negative  
biasa saja, tidak ada yang spesial dar   negative   negative   positive  


In [25]:
!pip install pipreqsnb

  Preparing metadata (setup.py) ... done
  Created wheel for pipreqsnb: filename=pipreqsnb-0.2.4-py3-none-any.whl size=4128 sha256=fe02237af684682b4fd6c32914c6f7fee72112983cf340548b994c500372223e
  Stored in directory: /root/.cache/pip/wheels/a6/f6/b8/5bd85cc669a8f4985ea596345127d33cf3569b037cf7e1194d
Successfully built pipreqsnb


In [28]:
!pipreqs "/content/drive/MyDrive/Colab Notebooks/Proyek_Sentimen" --scan-notebooks

Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
INFO: Successfully saved requirements file in /content/drive/MyDrive/Colab Notebooks/Proyek_Sentimen/requirements.txt
